# Bounding-box object insertion with FLUX.2 [klein] 4B

Approach: **bbox → mask → inpaint**.

1. Take a crop around the bbox (with context) so small boxes get more resolution.
2. Paint the bbox solid magenta and feed that image to klein as the edit input (optionally with a reference image of the object).
3. Ask klein to replace the magenta box with the object.
4. Blend the result back with a feathered bbox mask, so **pixels outside the box stay identical to the original**.

Klein has no native bbox/mask input, so steps 2–4 are what enforce the spatial constraint.

In [ ]:
import torch
from PIL import Image, ImageDraw, ImageFilter
import matplotlib.pyplot as plt
from diffusers import Flux2KleinPipeline
from diffusers.utils import load_image

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16

# Distilled 4B: 4 steps, guidance 1.0. (Apache 2.0, ~13 GB VRAM)
pipe = Flux2KleinPipeline.from_pretrained("black-forest-labs/FLUX.2-klein-4B", torch_dtype=dtype)

# Pick one:
pipe.to(device)                       # fastest, needs ~13+ GB VRAM
# pipe.enable_model_cpu_offload()     # use this instead if you run out of memory

## Helpers

In [ ]:
MASK_COLOR = (255, 0, 255)  # magenta placeholder, rarely appears in natural images


def get_crop_box(bbox, img_size, context_scale=2.5, min_side=256):
    """Crop around bbox: bbox size * context_scale per axis, clamped to the image."""
    x0, y0, x1, y1 = bbox
    W, H = img_size
    w, h = x1 - x0, y1 - y0
    cw = min(W, max(w * context_scale, min_side))
    ch = min(H, max(h * context_scale, min_side))
    cx, cy = (x0 + x1) / 2, (y0 + y1) / 2
    left = min(max(cx - cw / 2, 0), W - cw)
    top = min(max(cy - ch / 2, 0), H - ch)
    return tuple(int(round(v)) for v in (left, top, left + cw, top + ch))


def work_size(w, h, target=1024):
    """Scale so the long side is ~target; both sides multiples of 16."""
    s = target / max(w, h)
    return max(16, round(w * s / 16) * 16), max(16, round(h * s / 16) * 16)


def feather_mask(size, box, feather=12):
    """Soft rectangular mask ('L'): 255 inside the box, fading out over `feather` px."""
    m = Image.new("L", size, 0)
    x0, y0, x1, y1 = box
    ImageDraw.Draw(m).rectangle([x0 - feather, y0 - feather, x1 + feather, y1 + feather], fill=255)
    return m.filter(ImageFilter.GaussianBlur(feather / 2))


def default_prompt(obj, has_reference):
    if has_reference:
        return (f"Replace the solid magenta rectangle in the first image with the {obj} from the second image. "
                f"Match the lighting, perspective and shadows of the scene. Keep everything else unchanged.")
    return (f"Replace the solid magenta rectangle with {obj}. The object should fill the rectangle area, "
            f"with lighting, perspective and shadows matching the scene. Keep everything else unchanged.")


def insert_object(
    pipe, image, bbox, obj, reference=None, prompt=None,
    context_scale=2.5, target=1024, feather=12,
    steps=4, guidance=1.0, seed=0,
):
    """
    image : PIL image
    bbox  : (x0, y0, x1, y1) in pixels of `image`
    obj   : text description of what to generate, e.g. "a red ceramic teapot"
    reference : optional PIL image of the object to insert
    Returns (final_image, debug_dict)
    """
    image = image.convert("RGB")
    x0, y0, x1, y1 = bbox
    assert 0 <= x0 < x1 <= image.width and 0 <= y0 < y1 <= image.height, "bbox outside image"

    # 1) crop with context, resize to a Flux-friendly size
    crop_box = get_crop_box(bbox, image.size, context_scale)
    crop = image.crop(crop_box)
    cw, ch = crop.size
    ww, wh = work_size(cw, ch, target)
    sx, sy = ww / cw, wh / ch
    wb = ((x0 - crop_box[0]) * sx, (y0 - crop_box[1]) * sy,
          (x1 - crop_box[0]) * sx, (y1 - crop_box[1]) * sy)
    work = crop.resize((ww, wh), Image.LANCZOS)

    # 2) paint the bbox
    masked = work.copy()
    ImageDraw.Draw(masked).rectangle(wb, fill=MASK_COLOR)

    # 3) run klein (multi-reference edit: [masked scene, optional object reference])
    refs = [masked] + ([reference.convert("RGB")] if reference is not None else [])
    prompt = prompt or default_prompt(obj, reference is not None)
    gen = torch.Generator(device=pipe.device if hasattr(pipe, "device") else "cpu").manual_seed(seed)
    out = pipe(
        image=refs if len(refs) > 1 else refs[0],
        prompt=prompt, height=wh, width=ww,
        num_inference_steps=steps, guidance_scale=guidance, generator=gen,
    ).images[0].convert("RGB")
    if out.size != (ww, wh):
        out = out.resize((ww, wh), Image.LANCZOS)

    # 4) blend back at ORIGINAL resolution: only the (feathered) bbox changes
    mask = feather_mask((ww, wh), wb, feather)
    out_c = out.resize((cw, ch), Image.LANCZOS)
    mask_c = mask.resize((cw, ch), Image.LANCZOS)
    new_crop = Image.composite(out_c, crop, mask_c)
    final = image.copy()
    final.paste(new_crop, crop_box[:2])

    return final, dict(prompt=prompt, crop_box=crop_box, masked_input=masked, raw_output=out, mask=mask)


def show(images, titles=None, figsize=(16, 6)):
    fig, axs = plt.subplots(1, len(images), figsize=figsize)
    axs = [axs] if len(images) == 1 else axs
    for i, (ax, im) in enumerate(zip(axs, images)):
        ax.imshow(im, cmap="gray" if im.mode == "L" else None)
        ax.axis("off")
        if titles: ax.set_title(titles[i])
    plt.tight_layout(); plt.show()


def preview_bbox(image, bbox, color="red", width=4):
    im = image.convert("RGB").copy()
    ImageDraw.Draw(im).rectangle(bbox, outline=color, width=width)
    return im


def norm_to_px(bbox_norm, size):
    """(x0,y0,x1,y1) in 0..1 -> pixels."""
    W, H = size
    return (bbox_norm[0] * W, bbox_norm[1] * H, bbox_norm[2] * W, bbox_norm[3] * H)

## Load your image and set the bounding box

Use a URL, a local path, or (in Colab) `files.upload()`. `bbox` is `(x0, y0, x1, y1)` in **pixels**; use `norm_to_px` if yours is normalized.

In [ ]:
# image = load_image("https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/cat.png")
image = Image.open("scene.jpg").convert("RGB")   # <-- your image

bbox = (400, 300, 700, 600)                     # <-- your box (x0, y0, x1, y1), pixels
# bbox = norm_to_px((0.4, 0.5, 0.6, 0.8), image.size)

show([preview_bbox(image, bbox)], ["Input + bbox"], figsize=(8, 8))

## Run: text-described object

In [ ]:
result, dbg = insert_object(
    pipe, image, bbox,
    obj="a red ceramic teapot",
    seed=42,
)

print("Prompt:", dbg["prompt"])
show([image, dbg["masked_input"], dbg["raw_output"], result],
     ["Original", "Model input (crop)", "Raw model output (crop)", "Final (blended)"])
result.save("result.png")

## Optional: insert an object from a reference image

In [ ]:
# reference = Image.open("object.png").convert("RGB")   # ideally the object on a clean/plain background
# result, dbg = insert_object(pipe, image, bbox, obj="the teapot", reference=reference, seed=42)
# show([image, result], ["Original", "Final"])

## Tips

- **Seeds:** the box constrains *where*, not *what exactly*; try several seeds (`seed=0..7`) and pick the best.
- **Box too loose:** klein tends to fill the whole rectangle. Draw the box tight to the expected object extent.
- **Context:** raise `context_scale` (e.g. 3–4) if the object looks out of place; lower it (1.5–2) for tiny boxes where you want more detail.
- **Seams:** raise `feather` (e.g. 20–30) for visible edges; lower it if the box bleeds into neighbors.
- **Color bleed:** if magenta shows in the output, change `MASK_COLOR` to another color absent from the scene (e.g. `(0, 255, 0)`) and update the prompt wording.
- **Kwarg errors:** if your diffusers version rejects `guidance_scale`, remove it. For the *base* (non-distilled) model use `steps=50, guidance=4.0`.